Continuous Bag of Words (CBOW) is a neural network-based model used to predict a target word given its surrounding context words (window space). This model focuses on the context around a word to predict the word itself, which is useful for learning meaningful word representations. CBOW uses unsupervised learning, meaning it learns from large collections of text without requiring labeled data.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt

In [7]:
corpus = ['The cat sat on the mat',
          'The dog ran in the park',
          'The bird sang in the tree']
from sklearn.preprocessing import LabelEncoder
from collections import Counter
tokens = [sentence.lower().split() for sentence in corpus]
counter = Counter()
for sentence in tokens:
    counter.update(sentence)

vocab = {"<PAD>": 0, "<UNK>": 1}

for word in counter:
    vocab[word] = len(vocab)

print("Vocabulary:")
print(vocab)

sequences = []

for sentence in tokens:
    sequences.append([vocab[word] for word in sentence])

print("\nSequences:")
print(sequences)

Vocabulary:
{'<PAD>': 0, '<UNK>': 1, 'the': 2, 'cat': 3, 'sat': 4, 'on': 5, 'mat': 6, 'dog': 7, 'ran': 8, 'in': 9, 'park': 10, 'bird': 11, 'sang': 12, 'tree': 13}

Sequences:
[[2, 3, 4, 5, 2, 6], [2, 7, 8, 9, 2, 10], [2, 11, 12, 9, 2, 13]]


Neural networks only understand numbers.
<BR>
They cannot compute with:
<BR>
"cat"<BR>
"dog"<BR>
"bird"<BR>
<BR>
So we transform text like this:
<BR>
Sentence<BR>
     ↓<BR>
Words<BR>
     ↓<BR>
Vocabulary<BR>
     ↓<BR>
Integers<BR>
     ↓<BR>
Tensor<BR>
     ↓<BR>
Embedding Layer<BR>
     ↓<BR>
Neural Network<BR>
<BR>
This conversion from text to integer IDs is the first preprocessing step in almost every NLP pipeline.

In [8]:
vocab_size = len(vocab)
embedding_size = 10
window_size = 2


In [9]:
contexts = []
targets = []

for sequence in sequences:
    for i in range(window_size, len(sequence)-window_size):
        context = sequence[i-window_size:i] + sequence[i+1:i+window_size+1]
        target = sequence[i]

        contexts.append(context)
        targets.append(target)

print(contexts)
print(targets)
print(vocab)

X = np.array(contexts)
y = np.array(targets)

[[2, 3, 5, 2], [3, 4, 2, 6], [2, 7, 9, 2], [7, 8, 2, 10], [2, 11, 9, 2], [11, 12, 2, 13]]
[4, 5, 8, 9, 12, 9]
{'<PAD>': 0, '<UNK>': 1, 'the': 2, 'cat': 3, 'sat': 4, 'on': 5, 'mat': 6, 'dog': 7, 'ran': 8, 'in': 9, 'park': 10, 'bird': 11, 'sang': 12, 'tree': 13}


In [10]:
class CBOWDataset(Dataset):
    def __init__(self, contexts, targets):
        self.contexts = torch.tensor(contexts, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.contexts[idx], self.targets[idx]

dataset = CBOWDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [11]:
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        super(CBOWModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_size)
        self.linear = nn.Linear(embedding_size, vocab_size)

    def forward(self, inputs):
        embedded = self.embeddings(inputs)
        embedded_mean = embedded.mean(dim=1)
        out = self.linear(embedded_mean)
        return out

In [12]:
model = CBOWModel(vocab_size=vocab_size, embedding_size=embedding_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [60]:
epochs = 100
for epoch in range(epochs):
    total_loss = 0
    for context, target in dataloader:
        optimizer.zero_grad()
        output = model(context)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {total_loss/len(dataloader)}')

Epoch 0, Loss: 2.9802316466505846e-07
Epoch 10, Loss: 2.781549615823072e-07
Epoch 20, Loss: 2.781549663192588e-07
Epoch 30, Loss: 2.781549663192588e-07
Epoch 40, Loss: 2.781549615823072e-07
Epoch 50, Loss: 2.781549663192588e-07
Epoch 60, Loss: 2.781549663192588e-07
Epoch 70, Loss: 2.781549663192588e-07
Epoch 80, Loss: 2.7815496868773454e-07
Epoch 90, Loss: 2.5828675613108015e-07
